# CSE 151B: Homework 2 Coding
## PyTorch Implementation

Using PyTorch’s `Sequential` model class, build a deep convolutional network to classify handwritten digits in MNIST.

You are only allowed to use the following in your model design:
- Linear Layers
- Conv2D
- MaxPool2D
- BatchNorm2D
- Dropout Layers
- ReLU and Softmax
- Flatten

Your goal is to build a model that achieves **test accuracy ≥ 0.985** with fewer than 1 million parameters.

**Warning**: The modules in your Sequential network should *only* consist of `nn` objects! That means you should not be using `torch.nn.functional` modules or lambda expressions in your Sequential block. Leaving functional/lambda expressions in your model code will result in no credit!

This notebook provides a skeleton layout for you. You may use whatever parts of this notebook you deem necessary; there is no need for you to adhere to the structure. However, during submission, you must carefully follow the zip file formatting as requested; see the bottom of the notebook.

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [23]:
def get_data_loaders(batch_size) -> tuple[DataLoader, DataLoader]:
    '''
    Return the training and testing MNIST dataloaders.
    '''
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    # ========================
    # TODO: create dataloaders
    # ========================

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, test_loader

get_data_loaders.__defaults__ = (32,)


In [24]:
def build_model(dropout_prob=0.5) -> nn.Module:
    model = nn.Sequential(
        # ==========================
        # TODO: fill in architecture
        # ==========================

        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),

        nn.Conv2d(32, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),

        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(dropout_prob),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(dropout_prob),

        nn.Flatten(),

        nn.Linear(64 * 7 * 7, 128),
        nn.ReLU(),
        nn.Dropout(dropout_prob),

        nn.Linear(128, 10)
    )
    return model


In [25]:
def check_params():
    model = build_model()
    print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

In [26]:
def train(model, optimizer, criterion, train_loader, n_epochs = 1):
    '''
    Train the model for `n_epochs` epochs. Returns none (model is modified in place)
    '''
    model.train()
    # =====================
    # TODO: train the model
    # =====================

    device = next(model.parameters()).device

    for epoch in range(n_epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        avg_loss = running_loss / total
        train_acc = correct / total

        print(f"Epoch {epoch + 1}/{n_epochs} - Loss: {avg_loss:.4f} - Train accuracy: {train_acc:.4f}")

In [27]:
def test(model, test_loader):
    '''
    Tests the model. Returns none (you should print the accuracy)
    '''
    model.eval()
    # =================================
    # TODO: evaluate the model accuracy
    # =================================

    device = next(model.parameters()).device

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Test accuracy: {accuracy:.4f} ({correct}/{total})")

In [28]:
# try 10 different dropout values
train_loader, test_loader = get_data_loaders()
criterion = nn.CrossEntropyLoss() # TODO: use a criterion/loss function
dropout_values = [i / 10 for i in range(10)]

for p in dropout_values:
    model = build_model(dropout_prob=p)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4) # TODO: use an optimizer
    train(model, optimizer, criterion, train_loader)
    test(model, test_loader)
    torch.save(model, f'hw2_dropout_{p}.pt')

100.0%
100.0%
100.0%
100.0%


Epoch 1/1 - Loss: 0.0967 - Train accuracy: 0.9701
Test accuracy: 0.9854 (9854/10000)
Epoch 1/1 - Loss: 0.1114 - Train accuracy: 0.9655
Test accuracy: 0.9820 (9820/10000)
Epoch 1/1 - Loss: 0.1329 - Train accuracy: 0.9587
Test accuracy: 0.9893 (9893/10000)
Epoch 1/1 - Loss: 0.1604 - Train accuracy: 0.9494
Test accuracy: 0.9880 (9880/10000)
Epoch 1/1 - Loss: 0.2047 - Train accuracy: 0.9371
Test accuracy: 0.9874 (9874/10000)
Epoch 1/1 - Loss: 0.3107 - Train accuracy: 0.9033
Test accuracy: 0.9814 (9814/10000)
Epoch 1/1 - Loss: 0.4371 - Train accuracy: 0.8578
Test accuracy: 0.9808 (9808/10000)
Epoch 1/1 - Loss: 0.7177 - Train accuracy: 0.7462
Test accuracy: 0.9751 (9751/10000)
Epoch 1/1 - Loss: 1.3336 - Train accuracy: 0.5045
Test accuracy: 0.9553 (9553/10000)
Epoch 1/1 - Loss: 2.1618 - Train accuracy: 0.1788
Test accuracy: 0.5124 (5124/10000)


In [30]:
# find your best model, and train it for 10 epochs
best_p = 0.2 # TODO: fill in your best probability
model = build_model(dropout_prob=best_p)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4) # TODO: use an optimizer
train(model, optimizer, criterion, train_loader, n_epochs = 10)
test(model, test_loader)
torch.save(model, "hw2_model.pt")

Epoch 1/10 - Loss: 0.1396 - Train accuracy: 0.9568
Epoch 2/10 - Loss: 0.0644 - Train accuracy: 0.9813
Epoch 3/10 - Loss: 0.0498 - Train accuracy: 0.9847
Epoch 4/10 - Loss: 0.0446 - Train accuracy: 0.9857
Epoch 5/10 - Loss: 0.0408 - Train accuracy: 0.9871
Epoch 6/10 - Loss: 0.0372 - Train accuracy: 0.9887
Epoch 7/10 - Loss: 0.0349 - Train accuracy: 0.9886
Epoch 8/10 - Loss: 0.0319 - Train accuracy: 0.9901
Epoch 9/10 - Loss: 0.0319 - Train accuracy: 0.9900
Epoch 10/10 - Loss: 0.0293 - Train accuracy: 0.9910
Test accuracy: 0.9926 (9926/10000)


# Submission Instructions

Zip all of your **code** and **model .pt files** into one file, and submit on Gradescope to the respective submission.